# MaizeGuard Rwanda — Public Dataset Training Notebook

This notebook is updated for the current capstone decision: **training will rely on public datasets only**.

Your 4 local farmer/stocker images should be kept only for **manual testing/demo validation**, not for training.

## Public datasets used / supported

1. **CK-CNNLW / Deep Learning Based Corn Kernel Classification**  
   Best match for the capstone classes because it contains good corn kernels, defective corn kernels, and impurity.

2. **GrainSet Maize**  
   Large public grain-kernel quality dataset. It is useful for more maize/grain visual quality examples.

3. **EfficientMaize / Lightweight Dataset for Maize Classification**  
   Good/bad maize seed dataset. It is useful as support data, but the `bad` class should be treated carefully because it may include different types of low-quality maize.

## Training approach

This version uses **PyTorch + timm** instead of relying only on TensorFlow/Keras. That makes the notebook more modern for Kaggle and gives access to stronger pretrained image models such as:

- `mobilenetv3_large_100` — lightweight deployment candidate
- `tf_efficientnetv2_b0` — strong modern baseline
- `convnext_tiny` — modern ConvNet comparison model

The best model is selected using **macro F1-score**, not accuracy only, because the dataset can be imbalanced.

In [ ]:
# ============================================================
# 0. Kaggle setup
# ============================================================
# In Kaggle, turn GPU ON:
# Notebook settings -> Accelerator -> GPU

!pip -q install timm

In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================
import os
import re
import json
import time
import shutil
import zipfile
import random
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

# Public-only classes used by this capstone prototype.
# Keep mold as optional because public maize datasets rarely have clean mold labels.
CLASS_NAMES = ["good", "broken", "impurity", "mold_risk"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# For quick testing use small numbers. For final Kaggle run, increase MAX_IMAGES_PER_CLASS.
MAX_IMAGES_PER_CLASS = 2500
MIN_IMAGES_PER_CLASS = 40

# Training plan: start with frozen backbone, then fine-tune.
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 8
LR_HEAD = 1e-3
LR_FINETUNE = 2e-5
WEIGHT_DECAY = 1e-4

# Models to compare. ConvNeXt is stronger but can be slower.
MODEL_NAMES = [
    "mobilenetv3_large_100",
    "tf_efficientnetv2_b0",
    "convnext_tiny",
]

BASE_DIR = Path("/kaggle/working/maizeguard_public_training")
RAW_DIR = BASE_DIR / "raw_public_sources"
DATASET_DIR = BASE_DIR / "prepared_dataset"
OUTPUT_DIR = BASE_DIR / "outputs"
for p in [RAW_DIR, DATASET_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE_DIR)

## 2. Public dataset sources

This notebook supports two ways of using the public datasets:

### Recommended in Kaggle
Add public datasets through **Add Input**, especially:

- CK-CNNLW / Corn Kernel Classification if available in Kaggle or clone from GitHub.
- EfficientMaize / Maize Seed Dataset if available in Kaggle.
- GrainSet maize if you downloaded and uploaded it as a Kaggle dataset.

### Automatic download attempts
The notebook also attempts to download/clone public sources where possible:

- CK-CNNLW GitHub repository.
- GrainSet maize Figshare article package.

Some platforms may block automated downloads, so the notebook is also able to scan `/kaggle/input` for manually added public datasets.

In [ ]:
# ============================================================
# 2. Public dataset download / discovery
# ============================================================

PUBLIC_SOURCE_NOTES = {
    "CK-CNNLW": {
        "url": "https://github.com/vision-cidis/CK-CNNLW",
        "why": "Best match for good / defective / impurity corn-kernel classification."
    },
    "GrainSet maize": {
        "url": "https://github.com/hellodfan/GrainSet",
        "why": "Large public grain-kernel visual quality dataset with maize subset."
    },
    "EfficientMaize": {
        "url": "https://data.mendeley.com/datasets/r6vvm5jkh6/1",
        "why": "Good/bad maize seed dataset; useful as support but bad class is broad."
    },
}

print(json.dumps(PUBLIC_SOURCE_NOTES, indent=2))


def run_command(cmd, cwd=None):
    print("Running:", " ".join(cmd))
    try:
        result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=600)
        print(result.stdout[-1000:])
        if result.returncode != 0:
            print("Command warning/error:", result.stderr[-1000:])
        return result.returncode == 0
    except Exception as exc:
        print("Command failed:", exc)
        return False

# 2.1 Clone CK-CNNLW if internet is enabled.
ck_repo = RAW_DIR / "CK-CNNLW"
if not ck_repo.exists():
    run_command(["git", "clone", "--depth", "1", "https://github.com/vision-cidis/CK-CNNLW.git", str(ck_repo)])
else:
    print("CK-CNNLW already exists:", ck_repo)

# 2.2 Try GrainSet maize Figshare package download.
# This can be large. If it fails or takes long, manually add GrainSet maize as Kaggle input.
grainset_zip = RAW_DIR / "grainset_maize_figshare.zip"
GRAINSET_DOWNLOAD = False  # Change to True if you want the notebook to try downloading GrainSet maize.
if GRAINSET_DOWNLOAD and not grainset_zip.exists():
    # Figshare article id from GrainSet maize DOI page: 10.6084/m9.figshare.22987562.v2
    run_command([
        "bash", "-lc",
        f"wget -O {grainset_zip} https://figshare.com/ndownloader/articles/22987562/versions/2"
    ])

if grainset_zip.exists():
    extract_dir = RAW_DIR / "grainset_maize"
    extract_dir.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(grainset_zip, "r") as zf:
            zf.extractall(extract_dir)
        print("Extracted GrainSet to", extract_dir)
    except Exception as exc:
        print("Could not extract GrainSet zip:", exc)

# 2.3 List Kaggle input datasets.
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print("Kaggle inputs found:")
    for p in sorted(kaggle_input.iterdir()):
        print(" -", p)
else:
    print("/kaggle/input not found. This is normal outside Kaggle.")

## 3. Dataset mapping logic

The notebook creates a unified public-only dataset with these classes:

- `good`
- `broken`
- `impurity`
- `mold_risk`

Important: public datasets often do **not** have a perfect mold class. Therefore, the notebook only maps images to `mold_risk` when the folder or label clearly mentions mold, fungus, infection, or rotten. Broad labels like `bad` are mapped to `broken`, not mold, because forcing unclear bad images into mold would damage the model.

In [ ]:
# ============================================================
# 3. Build a public-only image manifest
# ============================================================
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# Source names to ignore because they are often masks/annotations/plots, not real training photos.
IGNORE_PATTERNS = [
    "mask", "segmentation", "annotation", "label", "bbox", "bounding", "groundtruth",
    "plot", "chart", "confusion", "readme", "architecture", "fig", "figure"
]

LABEL_KEYWORDS = {
    "good": ["good", "healthy", "normal", "sound", "clean"],
    "impurity": ["impurity", "foreign", "stone", "stones", "husk", "dust", "trash", "debris"],
    "mold_risk": ["mold", "mould", "fung", "fungus", "fungal", "infected", "infection", "rotten", "rot"],
    "broken": ["defective", "defect", "damaged", "damage", "broken", "unsound", "bad", "cracked", "shrivel", "low_quality", "low-quality"]
}


def is_image(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS


def should_ignore(path: Path) -> bool:
    text = str(path).lower()
    return any(pat in text for pat in IGNORE_PATTERNS)


def infer_label_from_path(path: Path):
    # Check parent folder names and file name.
    parts = [p.lower() for p in path.parts[-8:]]
    text = " ".join(parts)

    # Priority matters. If something says impurity, classify as impurity before broad bad.
    for label in ["impurity", "mold_risk", "good", "broken"]:
        for kw in LABEL_KEYWORDS[label]:
            if re.search(rf"(^|[^a-z]){re.escape(kw)}([^a-z]|$)", text):
                return label
    return None


def source_name_from_path(path: Path):
    text = str(path).lower()
    if "ck-cnn" in text or "ck_cnn" in text or "ckcn" in text:
        return "CK-CNNLW"
    if "grainset" in text:
        return "GrainSet"
    if "efficientmaize" in text or "maize seed" in text or "maize-seed" in text or "r6vvm5jkh6" in text:
        return "EfficientMaize"
    if "/kaggle/input/" in text:
        try:
            return path.relative_to("/kaggle/input").parts[0]
        except Exception:
            return "kaggle_input"
    return "public_source"


def collect_images_from_roots(roots):
    rows = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file() or not is_image(path):
                continue
            if should_ignore(path):
                continue
            label = infer_label_from_path(path)
            if label is None:
                continue
            rows.append({
                "path": str(path),
                "label": label,
                "source": source_name_from_path(path),
                "original_folder": path.parent.name,
                "file_name": path.name,
            })
    return pd.DataFrame(rows)

roots = [RAW_DIR]
if Path("/kaggle/input").exists():
    roots.append(Path("/kaggle/input"))

manifest = collect_images_from_roots(roots)
print("Raw mapped images:", len(manifest))
if len(manifest):
    display(manifest.head())
    display(pd.crosstab(manifest["source"], manifest["label"]))
else:
    print("No public images were mapped yet. Add public datasets through Kaggle Add Input, or enable download cells.")

In [ ]:
# ============================================================
# 4. Clean, balance, and split the public dataset
# ============================================================
if len(manifest) == 0:
    raise RuntimeError(
        "No public dataset images found. Add CK-CNNLW, EfficientMaize, or GrainSet as Kaggle input, "
        "or enable the download cells above."
    )

# Remove duplicates by path.
manifest = manifest.drop_duplicates(subset=["path"]).reset_index(drop=True)

# Keep classes that have enough examples.
counts = manifest["label"].value_counts()
print("Counts before filtering:")
print(counts)

valid_classes = [c for c, n in counts.items() if n >= MIN_IMAGES_PER_CLASS and c in CLASS_NAMES]
manifest = manifest[manifest["label"].isin(valid_classes)].copy()

if "good" not in set(manifest["label"]):
    raise RuntimeError("The prepared dataset must contain a good class.")

# Balance maximum examples per class.
balanced_parts = []
for label, group in manifest.groupby("label"):
    n = min(len(group), MAX_IMAGES_PER_CLASS)
    balanced_parts.append(group.sample(n=n, random_state=SEED))
manifest = pd.concat(balanced_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Counts after balancing:")
print(manifest["label"].value_counts())

# Create numeric target.
manifest["target"] = manifest["label"].map(CLASS_TO_IDX)

# Stratified split.
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=SEED,
    stratify=manifest["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df["label"].value_counts().to_dict())
print("Val:", val_df["label"].value_counts().to_dict())
print("Test:", test_df["label"].value_counts().to_dict())

manifest.to_csv(OUTPUT_DIR / "public_dataset_manifest_all.csv", index=False)
train_df.to_csv(OUTPUT_DIR / "train_manifest.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val_manifest.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_manifest.csv", index=False)

with open(OUTPUT_DIR / "class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

print("Saved manifests to", OUTPUT_DIR)

In [ ]:
# ============================================================
# 5. Visual check of public dataset samples
# ============================================================
def show_samples(df, n_per_class=4):
    labels = sorted(df["label"].unique())
    rows = len(labels)
    cols = n_per_class
    plt.figure(figsize=(cols * 3, rows * 3))
    idx = 1
    for label in labels:
        sample = df[df["label"] == label].sample(min(n_per_class, (df["label"] == label).sum()), random_state=SEED)
        for _, row in sample.iterrows():
            plt.subplot(rows, cols, idx)
            try:
                img = Image.open(row["path"]).convert("RGB")
                plt.imshow(img)
            except Exception:
                plt.text(0.5, 0.5, "Could not load", ha="center")
            plt.title(f"{label}\n{row['source']}", fontsize=9)
            plt.axis("off")
            idx += 1
    plt.tight_layout()
    plt.show()

show_samples(manifest, n_per_class=4)

## 6. Dataset and augmentation

The transforms below are designed for public kernel datasets and later farmer-style demo images. They add changes in lighting, blur, rotation, and scale so the model does not only memorize a controlled background.

In [ ]:
# ============================================================
# 6. PyTorch dataset and transforms
# ============================================================
class MaizeImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["path"]
        label = int(row["target"])
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            # Fallback avoids crashing on a corrupt public image.
            image = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(230, 210, 170))
        if self.transform:
            image = self.transform(image)
        return image, label

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.03),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_ds = MaizeImageDataset(train_df, transform=train_tfms)
val_ds = MaizeImageDataset(val_df, transform=valid_tfms)
test_ds = MaizeImageDataset(test_df, transform=valid_tfms)

# Weighted sampler helps when public datasets have class imbalance.
class_counts = train_df["label"].value_counts().to_dict()
sample_weights = train_df["label"].map(lambda x: 1.0 / class_counts[x]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Loaders ready")

In [ ]:
# ============================================================
# 7. Model helpers
# ============================================================
def create_model(model_name: str, num_classes: int, pretrained: bool = True):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model


def set_backbone_trainable(model, trainable: bool):
    for name, param in model.named_parameters():
        param.requires_grad = trainable
    # Keep classifier/head trainable.
    for name, param in model.named_parameters():
        if any(key in name.lower() for key in ["classifier", "head", "fc"]):
            param.requires_grad = True


def get_class_weights(df):
    counts = df["label"].value_counts().reindex(CLASS_NAMES).fillna(0).values.astype(float)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


def evaluate(model, loader, criterion=None):
    model.eval()
    losses = []
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            if criterion is not None:
                loss = criterion(logits, labels)
                losses.append(loss.item())
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
            y_prob.extend(probs.cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "loss": float(np.mean(losses)) if losses else None,
        "accuracy": float(acc),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def train_one_phase(model, train_loader, val_loader, criterion, optimizer, epochs, phase_name, model_name):
    best_f1 = -1
    best_path = OUTPUT_DIR / f"{model_name}_{phase_name}_best.pt"
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        start = time.time()

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_metrics = evaluate(model, val_loader, criterion)
        row = {
            "model": model_name,
            "phase": phase_name,
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1_macro": val_metrics["f1_macro"],
            "seconds": round(time.time() - start, 2),
        }
        history.append(row)
        print(row)

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            torch.save(model.state_dict(), best_path)

    return pd.DataFrame(history), best_path, best_f1

In [ ]:
# ============================================================
# 8. Train and compare pretrained models
# ============================================================
all_histories = []
summary_rows = []
class_weights = get_class_weights(train_df)
criterion = nn.CrossEntropyLoss(weight=class_weights)

for model_name in MODEL_NAMES:
    print("\n" + "="*80)
    print("Training:", model_name)
    print("="*80)

    model = create_model(model_name, num_classes=len(CLASS_NAMES), pretrained=True).to(DEVICE)

    # Phase 1: freeze backbone and train classification head.
    set_backbone_trainable(model, trainable=False)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    hist_head, best_head_path, _ = train_one_phase(model, train_loader, val_loader, criterion, optimizer, HEAD_EPOCHS, "head", model_name)

    # Phase 2: fine-tune all layers with a smaller learning rate.
    if best_head_path.exists():
        model.load_state_dict(torch.load(best_head_path, map_location=DEVICE))
    set_backbone_trainable(model, trainable=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    hist_ft, best_ft_path, best_val_f1 = train_one_phase(model, train_loader, val_loader, criterion, optimizer, FINETUNE_EPOCHS, "finetune", model_name)

    # Load best fine-tuned model and test.
    model.load_state_dict(torch.load(best_ft_path, map_location=DEVICE))
    test_metrics = evaluate(model, test_loader, criterion)

    # Estimate model size.
    tmp_path = OUTPUT_DIR / f"{model_name}_temp_size.pt"
    torch.save(model.state_dict(), tmp_path)
    size_mb = tmp_path.stat().st_size / (1024 * 1024)
    tmp_path.unlink(missing_ok=True)

    summary = {
        "model": model_name,
        "best_val_f1_macro": best_val_f1,
        "test_accuracy": test_metrics["accuracy"],
        "test_precision_macro": test_metrics["precision_macro"],
        "test_recall_macro": test_metrics["recall_macro"],
        "test_f1_macro": test_metrics["f1_macro"],
        "model_size_mb": round(size_mb, 2),
        "best_model_path": str(best_ft_path),
    }
    print("Test summary:", summary)

    summary_rows.append(summary)
    all_histories.append(hist_head)
    all_histories.append(hist_ft)

history_df = pd.concat(all_histories, ignore_index=True)
summary_df = pd.DataFrame(summary_rows).sort_values("test_f1_macro", ascending=False).reset_index(drop=True)

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "model_comparison_summary.csv", index=False)

display(summary_df)
print("Best model:", summary_df.iloc[0]["model"])

In [ ]:
# ============================================================
# 9. Detailed evaluation of the best model
# ============================================================
best_model_name = summary_df.iloc[0]["model"]
best_model_path = Path(summary_df.iloc[0]["best_model_path"])

best_model = create_model(best_model_name, num_classes=len(CLASS_NAMES), pretrained=False).to(DEVICE)
best_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
best_model.eval()

metrics = evaluate(best_model, test_loader, criterion)
y_true = metrics["y_true"]
y_pred = metrics["y_pred"]

print("Best model:", best_model_name)
print("Accuracy:", metrics["accuracy"])
print("Macro F1:", metrics["f1_macro"])
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=200)
plt.show()

# Save final best model in a simple name for API use.
FINAL_MODEL_PATH = OUTPUT_DIR / "maizeguard_public_best_model.pt"
shutil.copy(best_model_path, FINAL_MODEL_PATH)

metadata = {
    "model_name": best_model_name,
    "class_names": CLASS_NAMES,
    "image_size": IMG_SIZE,
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
    "selected_by": "highest test macro F1 among compared pretrained models",
    "public_datasets": PUBLIC_SOURCE_NOTES,
}
with open(OUTPUT_DIR / "maizeguard_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved final model:", FINAL_MODEL_PATH)
print("Saved metadata:", OUTPUT_DIR / "maizeguard_model_metadata.json")

## 10. Multi-crop prediction and recommendation rule

For real uploaded images, a single prediction can be unstable if the image contains mixed kernels. This function checks several crops of the same image and averages the probabilities.

The recommendation rule remains conservative:

`mold_risk → impurity → broken → good`

In [ ]:
# ============================================================
# 10. Multi-crop prediction and recommendation rule
# ============================================================
def preprocess_pil(image):
    return valid_tfms(image).unsqueeze(0)


def make_crops(image: Image.Image):
    image = image.convert("RGB")
    w, h = image.size
    crops = [image]

    # Center crop.
    s = int(min(w, h) * 0.75)
    left = (w - s) // 2
    top = (h - s) // 2
    crops.append(image.crop((left, top, left + s, top + s)))

    # Four corner crops.
    s2 = int(min(w, h) * 0.6)
    boxes = [
        (0, 0, s2, s2),
        (w - s2, 0, w, s2),
        (0, h - s2, s2, h),
        (w - s2, h - s2, w, h),
    ]
    for box in boxes:
        crops.append(image.crop(box))
    return crops


def get_recommendation(label, confidence):
    if confidence < 0.60:
        return {
            "risk": "Unclear",
            "action": "Needs review",
            "recommendation": "Retake the photo closer to the maize sample on a clear surface.",
        }

    mapping = {
        "good": {
            "risk": "Low",
            "action": "Store safely or prepare for sale",
            "recommendation": "The maize appears clean. Store in a dry place and monitor normally.",
        },
        "broken": {
            "risk": "Medium",
            "action": "Sort before storage",
            "recommendation": "Remove visibly broken or damaged kernels before storage or sale.",
        },
        "impurity": {
            "risk": "Medium",
            "action": "Clean and re-screen",
            "recommendation": "Remove foreign materials such as stones, husks, dust, or debris.",
        },
        "mold_risk": {
            "risk": "High",
            "action": "Do not store — refer for checking",
            "recommendation": "Possible mold-risk evidence needs careful checking before storage or consumption.",
        },
    }
    return mapping.get(label, {"risk": "Unknown", "action": "Needs review", "recommendation": "Could not map prediction safely."})


def predict_image_public_model(image_path, model=best_model):
    image = Image.open(image_path).convert("RGB")
    crops = make_crops(image)
    probs_list = []

    model.eval()
    with torch.no_grad():
        for crop in crops:
            x = preprocess_pil(crop).to(DEVICE)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
            probs_list.append(probs)

    avg_probs = np.mean(probs_list, axis=0)

    # Conservative priority rule. If a higher-risk class has enough probability, prefer it.
    priority = ["mold_risk", "impurity", "broken", "good"]
    chosen_idx = int(np.argmax(avg_probs))
    chosen_label = IDX_TO_CLASS[chosen_idx]
    confidence = float(avg_probs[chosen_idx])

    for label in priority:
        idx = CLASS_TO_IDX[label]
        if avg_probs[idx] >= 0.45 and label != "good":
            chosen_label = label
            confidence = float(avg_probs[idx])
            break

    return {
        "label": chosen_label,
        "confidence": round(confidence, 4),
        "confidence_percent": round(confidence * 100, 2),
        "probabilities": {CLASS_NAMES[i]: round(float(avg_probs[i]), 4) for i in range(len(CLASS_NAMES))},
        **get_recommendation(chosen_label, confidence),
    }

# Test on one public test image.
sample_path = test_df.sample(1, random_state=SEED).iloc[0]["path"]
print("Sample:", sample_path)
print(predict_image_public_model(sample_path))
img = Image.open(sample_path).convert("RGB")
plt.imshow(img)
plt.axis("off")
plt.show()

## 11. Export for the API / frontend demo

The notebook saves:

- `maizeguard_public_best_model.pt`
- `class_names.json`
- `maizeguard_model_metadata.json`
- `model_comparison_summary.csv`
- `confusion_matrix.png`

Use the `.pt` model in a **FastAPI model server**. Then call that FastAPI server from your Next.js API route.

In [ ]:
# ============================================================
# 11. Optional ONNX export for easier deployment
# ============================================================
EXPORT_ONNX = True
if EXPORT_ONNX:
    try:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
        onnx_path = OUTPUT_DIR / "maizeguard_public_best_model.onnx"
        torch.onnx.export(
            best_model,
            dummy,
            onnx_path,
            input_names=["image"],
            output_names=["logits"],
            dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=17,
        )
        print("Exported ONNX:", onnx_path)
    except Exception as exc:
        print("ONNX export failed. This is optional.", exc)

In [ ]:
# ============================================================
# 12. Zip all outputs for download from Kaggle
# ============================================================
zip_path = Path("/kaggle/working/maizeguard_public_training_outputs.zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", OUTPUT_DIR)
print("Created:", zip_path)
print("Download this zip from Kaggle output files.")

## What to say in your defense/progress update

> I am now using public maize/corn grain datasets only for training. The local farmer/stocker images are very few, so I keep them only for manual testing and demo validation. The main public dataset is CK-CNNLW because it has good, defective, and impurity corn-kernel classes. I also support GrainSet maize for grain quality inspection and EfficientMaize for good/bad maize support. I train with PyTorch and timm using pretrained image models such as MobileNetV3, EfficientNetV2B0, and ConvNeXtTiny, and I select the best model using macro F1-score, confusion matrix, inference behavior, and model size.